What do i need to do here:
1. Set up the functions for isolating documents related to topics of interest.
2. Copy these to folders under 'sources'
3. Be sure to have each permutation documented.

What are we doing with these documents?
Load different configurations into AntConc to answer the following questions with regards to gender:
1. what is the crux of salvation during this period
2. who is the audience for the message? who are the messengers? and how is the message delivered?
3. how is faith articulated?

---

In order to focus in on the ways the culture of seventh-day adventism was gendered and how that was expressed in different periods of their history, I am using the topic model to isolate the documents under consideration to those related to the following topics: health, missions, spirtual development, and correspondence. 

Why these topics? Health and Missions are two correlated topics that capture the "work" denominational members undertook as expressions of their faith. This work was both personal/familial and evangelistic focused, as health began at home but was also a way to reach out beyond the community, particularly through the work of Kellogg and the growing network of Sanitariums established by the denomination, and missions included both travel to foreign lands as well as distributing literature to family and neighbors. Spiritual Development and Correspondence are two additional topics that provide access to the "softer" aspects of the religious culture - the ways community members were encouraged to understand themselves and the progression of their faith, the marks of a religious life well lived. As a group with a "materialist" understanding of the human person - the human is physical, in that there is no separate soul that is saved but an embodied person who is saved through resurrection, the actions of human beings were integrally connected to the process of salvation.

In these I am looking for the ways the religious culture was gendered and the connections between that expression and their beliefs about time. This will tie to both time and domesticity - two primary themes the dissertation is exploring.

## Functions

In [1]:
import os
import pandas as pd
from topictools import prepare_mallet_data

In [2]:
# Load in topic model data, metadata, and labels

%load_ext autoreload
%autoreload 2

data_dir = "/Users/jeriwieringa/Dissertation/data/"
model = prepare_mallet_data.MalletModel(os.path.join(data_dir, 
                                                     'model_outputs', 
                                                     'target_300_10.18497.state.gz'
                                                    ))
df = model.model()
params = model.extract_params()

In [3]:
# %load ../shared_elements/load_from_gsheets.py
# Load Labels

import gspread
from oauth2client.service_account import ServiceAccountCredentials
import pandas as pd

# Load data from Google Doc
scope = ['https://spreadsheets.google.com/feeds']
secrets = "/Users/jeriwieringa/Dissertation/dev/code/secrets/dissertation-881847769b13.json"
credentials = ServiceAccountCredentials.from_json_keyfile_name(secrets, scope)
gc = gspread.authorize(credentials)

doc_name = 'Topic Labels'
dts = gc.open(doc_name).sheet1

labels = pd.DataFrame(dts.get_all_records())

In [4]:
# %load ../shared_elements/load_metadata.py
# Load metadata

metadata = pd.read_csv(os.path.join(data_dir, "corpus_metadata", "meta.csv"), header=None).reset_index()
metadata.columns = ['doc_id', 'filename', 'citation', 'author', 
                    'periodical_name', 'volume', 'issue', 
                    'date', 'page', 'url','abrev']
metadata['date_formatted'] = pd.to_datetime(metadata['date'])

question to self: should I exclude YI?

In [5]:
# Aggregate data by topic and year

doc_topic = df.groupby(['#doc', 'topic'])['type'].count().reset_index(name="token_count")
doc_topic['topic'] = doc_topic['topic'].apply(pd.to_numeric)

dt = prepare_mallet_data.pivot_smooth_norm(doc_topic, params[0], '#doc', 'topic', 'token_count')
docs = dt.unstack().reset_index(name="topic_proportion")
docs = docs.merge(metadata, how='left', left_on="#doc", right_on="doc_id")\
        .merge(labels, how='left', left_on='topic', right_on="mallet_topic_id" )

In [6]:
docs[:2]

,topic,#doc,topic_proportion,doc_id,filename,citation,author,periodical_name,volume,issue,...,page,url,abrev,date_formatted,browser_topic_id,mallet_topic_id,topic_category,topic_label,topic_prevalence_overall,topic_words
0,0,0,0.000247,0,ADV18981201-V02-01-page12.txt,"Training School Advocate (Vol. 02.01) Dec 01, ...",NaN,Training School Advocate,02,01,...,12,http://documents.adventistarchives.org/Periodi...,ADV,1898-12-01,1,0,missions,Missions (Canvassing Work),0.02981,book canvasser order canvassing brother week s...
1,0,1,0.000279,1,ADV18981201-V02-01-page13.txt,"Training School Advocate (Vol. 02.01) Dec 01, ...",NaN,Training School Advocate,02,01,...,13,http://documents.adventistarchives.org/Periodi...,ADV,1898-12-01,1,0,missions,Missions (Canvassing Work),0.02981,book canvasser order canvassing brother week s...


In [7]:
# Function to get documents associated with particular topics or categories

def select_docs(topic_ids, start_year, end_year, threshold, 
                df = docs):
    """
    """
    #filter frame by years and threshold
    df = df[(df.date_formatted.dt.year >= start_year) & (df.date_formatted.dt.year <= end_year) & (df['topic_proportion'] >= threshold)]
    
    # check type of topic_id
    # for strings (category)
    if isinstance(topic_ids[0], str):
        # Get topic_ids for the category
        t_df = df[df['topic_category'].isin(topic_ids)]
    
    # for int or list (individual topics)
    else:
        t_df = df[df['topic'].isin(topic_ids)]
   
    # return list of documents that meet the criteria in form of tuple (category, topic_id, year, doc)
    select_t_df = t_df[['topic_category', 'topic', 'topic_label', 'date_formatted', 'filename']].copy()
    select_t_df['year'] = select_t_df['date_formatted'].dt.year
    return [tuple(x) for x in select_t_df.values]

In [8]:
#test functions
select_docs(['health'], 1848, 1855, 0.2)

[('health',
  98,
  'Health (Coffee Tea Tobacco)',
  Timestamp('1855-07-24 00:00:00'),
  'RH18550724-V07-02-page1.txt',
  1855),
 ('health',
  98,
  'Health (Coffee Tea Tobacco)',
  Timestamp('1855-07-24 00:00:00'),
  'RH18550724-V07-02-page2.txt',
  1855),
 ('health',
  98,
  'Health (Coffee Tea Tobacco)',
  Timestamp('1855-08-07 00:00:00'),
  'RH18550807-V07-03-page1.txt',
  1855),
 ('health',
  98,
  'Health (Coffee Tea Tobacco)',
  Timestamp('1855-08-07 00:00:00'),
  'RH18550807-V07-03-page2.txt',
  1855)]

In [9]:
# Copy files into associated folders under `sources` by year
# File structure: 
# study
#   cycle_id
#     topic_category
#       topic_id
#         year

import shutil

def check_dir(_dir):
    if not os.path.exists(_dir):
        os.makedirs(_dir)

def gather_docs(docs_list, cycle_id):
    """
    """
    input_dir = "/Users/jeriwieringa/Dissertation/sources/text/2017-04-Final-Corpus/"
    output_dir_base = "/Users/jeriwieringa/Dissertation/sources/study/"
    
    for each in docs_list:
        topic = " ".join([str(each[1]), each[2]])
        out_dir = os.path.join(output_dir_base, cycle_id, each[0], topic, str(each[5]))
        check_dir(out_dir)
                
        shutil.copy(os.path.join(input_dir, each[4]), os.path.join(out_dir, each[4]))

In [10]:
categories = ['health', 'missions', 'spiritual_growth', 'correspondence']

## Select Documents for Cycle A: 1849-1860

In [11]:
gather_docs(select_docs(categories, 1848, 1860, 0.2), 'cycleA_49-60')

## Select Documents for Cycle B: 1860-1885

Beginning in the 1860s, discussion of salvation within the denominational literature began to include an increased emphasis on practices and behaviors, particularly in relation to the subject of health. 

In [12]:
gather_docs(select_docs(categories, 1860, 1885, 0.2), 'cycleB_60-85')

## Select Documents for Cycle C: 1885-1920

In [13]:
gather_docs(select_docs(categories, 1885, 1920, 0.2), 'cycleB_85-20')